In [1]:
import sys

print("Python version:", sys.version)

Python version: 3.10.12 (main, Jan 26 2026, 14:55:28) [GCC 11.4.0]


In [2]:
!pip install pandas networkx ortools

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 6.0 MB/s eta 0:00:00m eta 0:00:010:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.6/27.6 MB 4.5 MB/s eta 0:00:00m eta 0:00:010:00:01m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.1/321.1 KB 4.7 MB/s eta 0:00:00m eta 0:00:01
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.27.3
    Uninstalling protobuf-5.27.3:
      Successfully uninstalled protobuf-5.27.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.18.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.31.1 which is incompatible.
streamlit 1.37.1 requires pillow<11,>=7.1.0, but you have pillow 11.1.0 which is incompatible.
streamlit 1.37.1 requires protobu

In [3]:
import pandas as pd
import networkx as nx
from ortools.sat.python import cp_model

print("All libraries are working!")

All libraries are working!


In [4]:
import os

print(os.listdir("data"))

['tracks.csv', 'trains.csv', 'stations.csv', 'Railway.ipynb']


In [2]:
import pandas as pd

stations = pd.read_csv("data/stations.csv")
tracks = pd.read_csv("data/tracks.csv")
trains = pd.read_csv("data/trains.csv")

print("STATIONS")
display(stations.head())

print("TRACKS")
display(tracks.head())

print("TRAINS")
display(trains.head())

STATIONS


,station_id,station_name,zone,latitude,longitude,station_type,platforms,daily_train_capacity
0,BZA,Vijayawada Junction,SCR,16.5193,80.6218,MAJOR_JUNCTION,10,350
1,KCC,Krishna Canal Junction,SCR,16.4812,80.6074,JUNCTION,3,120
2,MAG,Mangalagiri,SCR,16.4381,80.5574,PASSENGER_STATION,3,80
3,GNT,Guntur Junction,SCR,16.3067,80.4365,MAJOR_JUNCTION,7,180
4,TEL,Tenali Junction,SCR,16.2433,80.6483,JUNCTION,5,140


TRACKS


,track_id,from_station,to_station,distance_km,travel_time_min,capacity_trains_per_hour,track_type,status,maintenance_allowed,risk_score
0,T001,BZA,KCC,5.2,8,5,QUADRUPLE_ELECTRIFIED,AVAILABLE,YES,0.10
1,T002,KCC,MAG,7.8,10,4,DOUBLE_ELECTRIFIED,MAINTENANCE,YES,0.35
2,T003,MAG,GNT,21.3,22,3,DOUBLE_ELECTRIFIED,AVAILABLE,YES,0.15
3,T004,KCC,TEL,20.1,20,4,DOUBLE_ELECTRIFIED,AVAILABLE,YES,0.12
4,T005,TEL,GNT,25.4,26,3,DOUBLE_ELECTRIFIED,AVAILABLE,YES,0.18


TRAINS


,train_id,train_name,source_station,destination_station,scheduled_departure,scheduled_arrival,priority,train_type,passenger_count,average_speed_kmph,status
0,TR001,12711 Pinakini Express,BZA,OGL,06:00,07:45,HIGH,SUPERFAST,1250,75,ON_TIME
1,TR002,12712 Pinakini Express,OGL,BZA,16:15,18:10,HIGH,SUPERFAST,1210,72,ON_TIME
2,TR003,12727 Godavari Express,VSKP,BZA,05:15,11:30,HIGH,SUPERFAST,1400,68,DELAYED
3,TR004,12728 Godavari Express,BZA,VSKP,17:20,23:00,HIGH,SUPERFAST,1380,70,ON_TIME
4,TR005,17201 Golconda Express,GNT,KZJ,06:45,11:15,HIGH,EXPRESS,1100,55,ON_TIME


In [9]:
from src.predictor import evaluate_maintenance_block

result = evaluate_maintenance_block(
    train_count=15,
    track_utilization=85.0,
    block_duration=2.0,
    time=10,
    train_priority=2,
    previous_delay=10.0,
    diversion_penalty=20.0
)

print(result)

/home/adithya/.local/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.7.2 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/adithya/.local/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator RandomForestRegressor from version 1.7.2 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/adithya/.local/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.7.2 when using

{'predicted_delay_mins': 63.61, 'congestion_level': 'HIGH', 'impact_score': 145.41, 'recommendation': 'REJECTED'}


In [17]:
# ============================================================
# TASK 13: FINAL INTEGRATED RAILWAY OPTIMIZATION
# Member 2 ML + Member 3 NetworkX + OR-Tools
# ============================================================

import pandas as pd
import networkx as nx
from ortools.sat.python import cp_model
from src.predictor import evaluate_maintenance_block
import os


print("=" * 65)
print("AI-POWERED RAILWAY BLOCK PLANNING")
print("=" * 65)


# ============================================================
# 1. LOAD DATA
# ============================================================

stations = pd.read_csv("data/stations.csv")
tracks = pd.read_csv("data/tracks.csv")
trains = pd.read_csv("data/trains.csv")

print("\nData loaded successfully.")
print("Stations:", len(stations))
print("Tracks:", len(tracks))
print("Trains:", len(trains))


# ============================================================
# 2. CREATE RAILWAY GRAPH
# ============================================================

G = nx.Graph()

for _, row in stations.iterrows():

    G.add_node(
        row["station_id"],
        name=row["station_name"]
    )


for _, row in tracks.iterrows():

    G.add_edge(
        row["from_station"],
        row["to_station"],
        track_id=row["track_id"],
        distance=row["distance_km"],
        travel_time=row["travel_time_min"],
        capacity=row["capacity_trains_per_hour"],
        status=row["status"]
    )


print("\nRailway graph created.")
print("Graph stations:", G.number_of_nodes())
print("Graph tracks:", G.number_of_edges())


# ============================================================
# 3. FIND NORMAL ROUTES FOR ALL TRAINS
# ============================================================

normal_results = []

for _, train in trains.iterrows():

    source = train["source_station"]
    destination = train["destination_station"]

    try:

        route = nx.shortest_path(
            G,
            source=source,
            target=destination,
            weight="travel_time"
        )

        total_time = 0

        for i in range(len(route) - 1):

            total_time += G[
                route[i]
            ][
                route[i + 1]
            ]["travel_time"]


        normal_results.append({

            "train_id": train["train_id"],

            "source": source,

            "destination": destination,

            "normal_route":
                " → ".join(route),

            "travel_time_min":
                total_time,

            "priority":
                train["priority"]
        })


    except nx.NetworkXNoPath:

        normal_results.append({

            "train_id": train["train_id"],

            "source": source,

            "destination": destination,

            "normal_route": "No Route",

            "travel_time_min": None,

            "priority":
                train["priority"]
        })


normal_routes = pd.DataFrame(normal_results)

print(
    "\nNormal routes calculated:",
    len(normal_routes)
)


# ============================================================
# 4. FIND SUITABLE MAINTENANCE TRACK
# ============================================================

candidate_tracks = []

for u, v, data in list(G.edges(data=True)):

    affected_count = 0

    # --------------------------------------------------------
    # Count trains using this track
    # --------------------------------------------------------

    for _, train in normal_routes.iterrows():

        route = train["normal_route"]

        if route == "No Route":
            continue

        route_stations = route.split(" → ")

        for i in range(len(route_stations) - 1):

            a = route_stations[i]
            b = route_stations[i + 1]

            if (
                (a == u and b == v)
                or
                (a == v and b == u)
            ):

                affected_count += 1
                break


    # --------------------------------------------------------
    # Temporarily remove track
    # --------------------------------------------------------

    edge_data = data.copy()

    G.remove_edge(u, v)

    has_alternative = False
    alternative_time = None
    alternative_path = None

    try:

        path = nx.shortest_path(
            G,
            source=u,
            target=v,
            weight="travel_time"
        )

        has_alternative = True

        alternative_path = path

        alternative_time = 0

        for i in range(len(path) - 1):

            alternative_time += G[
                path[i]
            ][
                path[i + 1]
            ]["travel_time"]

    except nx.NetworkXNoPath:

        pass


    # --------------------------------------------------------
    # Restore track
    # --------------------------------------------------------

    G.add_edge(
        u,
        v,
        **edge_data
    )


    # --------------------------------------------------------
    # Save useful candidates
    # --------------------------------------------------------

    if affected_count > 0 and has_alternative:

        candidate_tracks.append({

            "from": u,

            "to": v,

            "affected_count":
                affected_count,

            "original_time":
                edge_data["travel_time"],

            "alternative_time":
                alternative_time,

            "alternative_route":
                " → ".join(alternative_path),

            "capacity":
                edge_data["capacity"]
        })


# ============================================================
# 5. SELECT TRACK
# ============================================================

if len(candidate_tracks) == 0:

    print("\nNo suitable maintenance track found.")

else:

    candidate_df = pd.DataFrame(candidate_tracks)

    candidate_df = candidate_df.sort_values(
        by="affected_count",
        ascending=False
    )

    selected = candidate_df.iloc[0]

    blocked_from = selected["from"]
    blocked_to = selected["to"]


    print("\n" + "=" * 65)
    print("SELECTED MAINTENANCE TRACK")
    print("=" * 65)

    print(
        "Track:",
        blocked_from,
        "→",
        blocked_to
    )

    print(
        "Affected trains:",
        int(selected["affected_count"])
    )

    print(
        "Alternative route:",
        selected["alternative_route"]
    )

    print(
        "Alternative time:",
        selected["alternative_time"],
        "minutes"
    )


    # ========================================================
    # 6. FIND AFFECTED TRAINS
    # ========================================================

    affected_results = []

    for _, train in normal_routes.iterrows():

        route = train["normal_route"]

        if route == "No Route":
            continue

        route_stations = route.split(" → ")

        uses_track = False

        for i in range(len(route_stations) - 1):

            a = route_stations[i]
            b = route_stations[i + 1]

            if (
                (a == blocked_from and b == blocked_to)
                or
                (a == blocked_to and b == blocked_from)
            ):

                uses_track = True
                break


        if uses_track:

            affected_results.append({

                "train_id":
                    train["train_id"],

                "source":
                    train["source"],

                "destination":
                    train["destination"],

                "normal_route":
                    train["normal_route"],

                "normal_time_min":
                    train["travel_time_min"],

                "priority":
                    train["priority"]
            })


    affected_df = pd.DataFrame(affected_results)


    print("\n" + "=" * 65)
    print("AFFECTED TRAINS")
    print("=" * 65)

    print(
        "Number of affected trains:",
        len(affected_df)
    )

    display(affected_df)


    # ========================================================
    # 7. FIND ALTERNATIVE ROUTES
    # ========================================================

    alternative_results = []

    blocked_data = G[
        blocked_from
    ][
        blocked_to
    ].copy()

    G.remove_edge(
        blocked_from,
        blocked_to
    )


    for _, train in affected_df.iterrows():

        try:

            path = nx.shortest_path(
                G,
                source=train["source"],
                target=train["destination"],
                weight="travel_time"
            )

            alternative_time = 0

            for i in range(len(path) - 1):

                alternative_time += G[
                    path[i]
                ][
                    path[i + 1]
                ]["travel_time"]


            delay = (
                alternative_time
                -
                train["normal_time_min"]
            )


            alternative_results.append({

                "train_id":
                    train["train_id"],

                "source":
                    train["source"],

                "destination":
                    train["destination"],

                "normal_route":
                    train["normal_route"],

                "alternative_route":
                    " → ".join(path),

                "normal_time_min":
                    train["normal_time_min"],

                "alternative_time_min":
                    alternative_time,

                "additional_delay_min":
                    max(0, delay),

                "priority":
                    train["priority"]
            })


        except nx.NetworkXNoPath:

            alternative_results.append({

                "train_id":
                    train["train_id"],

                "source":
                    train["source"],

                "destination":
                    train["destination"],

                "normal_route":
                    train["normal_route"],

                "alternative_route":
                    "No Alternative Route",

                "normal_time_min":
                    train["normal_time_min"],

                "alternative_time_min":
                    None,

                "additional_delay_min":
                    None,

                "priority":
                    train["priority"]
            })


    # Restore blocked track

    G.add_edge(
        blocked_from,
        blocked_to,
        **blocked_data
    )


    alternative_df = pd.DataFrame(
        alternative_results,
        columns=[
            "train_id",
            "source",
            "destination",
            "normal_route",
            "alternative_route",
            "normal_time_min",
            "alternative_time_min",
            "additional_delay_min",
            "priority"
        ]
    )


    print("\n" + "=" * 65)
    print("ALTERNATIVE ROUTES")
    print("=" * 65)

    display(alternative_df)


    # ========================================================
    # 8. MAINTENANCE SLOTS
    # ========================================================

    maintenance_slots = [

        "09:00 - 11:00",

        "10:00 - 12:00",

        "11:00 - 13:00",

        "12:00 - 14:00",

        "13:00 - 15:00",

        "14:00 - 16:00",

        "15:00 - 17:00",

        "16:00 - 18:00"
    ]


    # ========================================================
    # 9. CALCULATE INPUTS FOR MEMBER 2 ML
    # ========================================================

    # IMPORTANT:
    # Member 2 uses:
    # 1 = LOW
    # 2 = MEDIUM
    # 3 = HIGH

    priority_map = {

        "LOW": 1,

        "MEDIUM": 2,

        "HIGH": 3
    }


    # --------------------------------------------------------
    # Track utilization
    # --------------------------------------------------------

    track_capacity = float(
        selected["capacity"]
    )

    if track_capacity <= 0:

        track_capacity = 10


    track_utilization = min(

        100,

        (
            len(affected_df)
            /
            track_capacity
        ) * 100
    )


    # --------------------------------------------------------
    # Average diversion penalty
    # --------------------------------------------------------

    if len(alternative_df) > 0:

        diversion_penalty = float(

            alternative_df[
                "additional_delay_min"
            ]
            .fillna(0)
            .mean()
        )

    else:

        diversion_penalty = 0.0


    # --------------------------------------------------------
    # Determine HIGHEST priority among affected trains
    #
    # LOW    = 1
    # MEDIUM = 2
    # HIGH   = 3
    #
    # Therefore use MAX, not MIN.
    # --------------------------------------------------------

    if len(affected_df) > 0:

        priority_values = []

        for priority in affected_df["priority"]:

            priority_values.append(

                priority_map.get(

                    str(priority).upper(),

                    2
                )
            )


        train_priority = max(
            priority_values
        )

    else:

        train_priority = 2


    print("\nML INPUTS")

    print(
        "Train count:",
        len(affected_df)
    )

    print(
        "Track utilization:",
        round(
            track_utilization,
            2
        ),
        "%"
    )

    print(
        "Block duration: 2 hours"
    )

    print(
        "Train priority:",
        train_priority
    )

    print(
        "Average diversion penalty:",
        round(
            diversion_penalty,
            2
        )
    )


    # ========================================================
    # 10. MEMBER 2 ML EVALUATION
    # ========================================================

    slot_results = []

    for slot in maintenance_slots:

        start_hour = int(
            slot.split(":")[0]
        )


        ml_result = evaluate_maintenance_block(

            train_count=
                len(affected_df),

            track_utilization=
                track_utilization,

            block_duration=2.0,

            time=start_hour,

            train_priority=
                train_priority,

            previous_delay=0.0,

            diversion_penalty=
                diversion_penalty
        )


        slot_results.append({

            "maintenance_slot":
                slot,

            "predicted_delay_min":
                ml_result[
                    "predicted_delay_mins"
                ],

            "congestion_level":
                ml_result[
                    "congestion_level"
                ],

            "impact_score":
                ml_result[
                    "impact_score"
                ],

            "recommendation":
                ml_result[
                    "recommendation"
                ]
        })


    slot_df = pd.DataFrame(
        slot_results
    )


    print("\n" + "=" * 65)
    print("MEMBER 2 ML MAINTENANCE EVALUATION")
    print("=" * 65)

    display(slot_df)


    # ========================================================
    # 11. OR-TOOLS OPTIMIZATION
    # ========================================================

    print("\n" + "=" * 65)
    print("OR-TOOLS OPTIMIZATION")
    print("=" * 65)


    model = cp_model.CpModel()

    slot_variables = []


    for i in range(
        len(maintenance_slots)
    ):

        slot_variables.append(

            model.NewBoolVar(
                f"slot_{i}"
            )
        )


    # --------------------------------------------------------
    # Exactly one slot
    # --------------------------------------------------------

    model.Add(
        sum(slot_variables) == 1
    )


    # --------------------------------------------------------
    # Impact score as optimization cost
    # --------------------------------------------------------

    integer_costs = [

        int(
            round(
                score * 100
            )
        )

        for score in slot_df[
            "impact_score"
        ]
    ]


    model.Minimize(

        sum(

            slot_variables[i]
            *
            integer_costs[i]

            for i in range(
                len(slot_variables)
            )
        )
    )


    solver = cp_model.CpSolver()

    status = solver.Solve(
        model
    )


    # ========================================================
    # 12. GET OPTIMAL SLOT
    # ========================================================

    if status in [

        cp_model.OPTIMAL,

        cp_model.FEASIBLE

    ]:

        best_index = 0

        for i in range(
            len(slot_variables)
        ):

            if solver.Value(
                slot_variables[i]
            ) == 1:

                best_index = i

                break

    else:

        best_index = 0


    best_slot = maintenance_slots[
        best_index
    ]

    best_impact = slot_df.iloc[
        best_index
    ]["impact_score"]

    best_delay = slot_df.iloc[
        best_index
    ]["predicted_delay_min"]

    best_congestion = slot_df.iloc[
        best_index
    ]["congestion_level"]

    best_recommendation = slot_df.iloc[
        best_index
    ]["recommendation"]


    # ========================================================
    # 13. CHECK SAFE OPTIONS
    # ========================================================

    approved_slots = slot_df[
        slot_df[
            "recommendation"
        ] == "APPROVED"
    ]


    caution_slots = slot_df[
        slot_df[
            "recommendation"
        ] == "CAUTION"
    ]


    if len(approved_slots) > 0:

        decision = (
            "APPROVED SLOT AVAILABLE"
        )


    elif len(caution_slots) > 0:

        decision = (
            "NO APPROVED SLOT - "
            "BEST CAUTION SLOT SELECTED"
        )


    else:

        decision = (
            "NO SAFE SLOT - "
            "LOWEST IMPACT SLOT SHOWN "
            "FOR REVIEW"
        )


    # ========================================================
    # 14. FINAL OPTIMAL PLAN
    # ========================================================

    final_plan = pd.DataFrame([{

        "blocked_track":
            f"{blocked_from} → {blocked_to}",

        "maintenance_slot":
            best_slot,

        "affected_trains":
            len(affected_df),

        "predicted_delay_min":
            best_delay,

        "congestion_level":
            best_congestion,

        "impact_score":
            best_impact,

        "ML_recommendation":
            best_recommendation,

        "system_decision":
            decision
    }])


    print("\n" + "=" * 65)
    print("⭐ FINAL OPTIMAL MAINTENANCE PLAN")
    print("=" * 65)

    display(final_plan)


    # ========================================================
    # 15. SAVE RESULTS
    # ========================================================

    os.makedirs(
        "results",
        exist_ok=True
    )


    normal_routes.to_csv(
        "results/normal_routes.csv",
        index=False
    )


    affected_df.to_csv(
        "results/affected_trains.csv",
        index=False
    )


    alternative_df.to_csv(
        "results/alternative_routes.csv",
        index=False
    )


    slot_df.to_csv(
        "results/maintenance_slots.csv",
        index=False
    )


    final_plan.to_csv(
        "results/final_optimal_plan.csv",
        index=False
    )



AI-POWERED RAILWAY BLOCK PLANNING

Data loaded successfully.
Stations: 25
Tracks: 42
Trains: 65

Railway graph created.
Graph stations: 25
Graph tracks: 28

Normal routes calculated: 65

SELECTED MAINTENANCE TRACK
Track: BZA → EE
Affected trains: 30
Alternative route: BZA → NZD → EE
Alternative time: 62 minutes

AFFECTED TRAINS
Number of affected trains: 30


,train_id,source,destination,normal_route,normal_time_min,priority
0,TR003,VSKP,BZA,VSKP → DVD → SLO → RJY → NDD → TDD → EE → BZA,305,HIGH
1,TR004,BZA,VSKP,BZA → EE → TDD → NDD → RJY → SLO → DVD → VSKP,305,HIGH
2,TR007,KZJ,VSKP,KZJ → DKJ → KMT → RYP → BZA → EE → TDD → NDD →...,472,HIGH
3,TR008,VSKP,KZJ,VSKP → DVD → SLO → RJY → NDD → TDD → EE → BZA ...,472,HIGH
4,TR015,BZA,VSKP,BZA → EE → TDD → NDD → RJY → SLO → DVD → VSKP,305,LOW
5,TR016,VSKP,BZA,VSKP → DVD → SLO → RJY → NDD → TDD → EE → BZA,305,LOW
6,TR017,BZA,VSKP,BZA → EE → TDD → NDD → RJY → SLO → DVD → VSKP,305,MEDIUM
7,TR018,VSKP,BZA,VSKP → DVD → SLO → RJY → NDD → TDD → EE → BZA,305,MEDIUM
8,TR024,BZA,VSKP,BZA → EE → TDD → NDD → RJY → SLO → DVD → VSKP,305,MEDIUM
9,TR026,RJY,BZA,RJY → NDD → TDD → EE → BZA,128,MEDIUM



ALTERNATIVE ROUTES


,train_id,source,destination,normal_route,alternative_route,normal_time_min,alternative_time_min,additional_delay_min,priority
0,TR003,VSKP,BZA,VSKP → DVD → SLO → RJY → NDD → TDD → EE → BZA,VSKP → DVD → SLO → RJY → NDD → TDD → EE → NZD ...,305,319,14,HIGH
1,TR004,BZA,VSKP,BZA → EE → TDD → NDD → RJY → SLO → DVD → VSKP,BZA → NZD → EE → TDD → NDD → RJY → SLO → DVD →...,305,319,14,HIGH
2,TR007,KZJ,VSKP,KZJ → DKJ → KMT → RYP → BZA → EE → TDD → NDD →...,KZJ → DKJ → KMT → RYP → BZA → NZD → EE → TDD →...,472,486,14,HIGH
3,TR008,VSKP,KZJ,VSKP → DVD → SLO → RJY → NDD → TDD → EE → BZA ...,VSKP → DVD → SLO → RJY → NDD → TDD → EE → NZD ...,472,486,14,HIGH
4,TR015,BZA,VSKP,BZA → EE → TDD → NDD → RJY → SLO → DVD → VSKP,BZA → NZD → EE → TDD → NDD → RJY → SLO → DVD →...,305,319,14,LOW
5,TR016,VSKP,BZA,VSKP → DVD → SLO → RJY → NDD → TDD → EE → BZA,VSKP → DVD → SLO → RJY → NDD → TDD → EE → NZD ...,305,319,14,LOW
6,TR017,BZA,VSKP,BZA → EE → TDD → NDD → RJY → SLO → DVD → VSKP,BZA → NZD → EE → TDD → NDD → RJY → SLO → DVD →...,305,319,14,MEDIUM
7,TR018,VSKP,BZA,VSKP → DVD → SLO → RJY → NDD → TDD → EE → BZA,VSKP → DVD → SLO → RJY → NDD → TDD → EE → NZD ...,305,319,14,MEDIUM
8,TR024,BZA,VSKP,BZA → EE → TDD → NDD → RJY → SLO → DVD → VSKP,BZA → NZD → EE → TDD → NDD → RJY → SLO → DVD →...,305,319,14,MEDIUM
9,TR026,RJY,BZA,RJY → NDD → TDD → EE → BZA,RJY → NDD → TDD → EE → NZD → BZA,128,142,14,MEDIUM



ML INPUTS
Train count: 30
Track utilization: 100 %
Block duration: 2 hours
Train priority: 3
Average diversion penalty: 14.0

MEMBER 2 ML MAINTENANCE EVALUATION


,maintenance_slot,predicted_delay_min,congestion_level,impact_score,recommendation
0,09:00 - 11:00,80.37,HIGH,204.74,REJECTED
1,10:00 - 12:00,80.26,HIGH,204.52,REJECTED
2,11:00 - 13:00,80.27,HIGH,204.54,REJECTED
3,12:00 - 14:00,80.27,HIGH,204.54,REJECTED
4,13:00 - 15:00,80.27,HIGH,204.54,REJECTED
5,14:00 - 16:00,80.27,HIGH,204.54,REJECTED
6,15:00 - 17:00,80.31,HIGH,204.62,REJECTED
7,16:00 - 18:00,80.34,HIGH,204.68,REJECTED



OR-TOOLS OPTIMIZATION

⭐ FINAL OPTIMAL MAINTENANCE PLAN


,blocked_track,maintenance_slot,affected_trains,predicted_delay_min,congestion_level,impact_score,ML_recommendation,system_decision
0,BZA → EE,10:00 - 12:00,30,80.26,HIGH,204.52,REJECTED,NO SAFE SLOT - LOWEST IMPACT SLOT SHOWN FOR RE...



Results saved successfully.
